# Load data

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from walinet.parameter_calibration.load_data import *

from walinet.parameter_calibration.compute_statistics import *

from walinet.parameter_calibration.pipeline_FWHM_SNR_shifts import *

from walinet.parameter_calibration.water_lipid_ratios import *

from walinet.parameter_calibration.plot_statistics import *

In [ ]:
bandwidth_hz = 939.85
nmr_frequency_hz = 123231706.0
water_ppm = 4.68

In [ ]:
SUBJECT_DIRS = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/ClimaX_Brisbane/Vol4/Res36x36",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/ClimaX_Brisbane/Vol4/Res50x50",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMAFIT_Vienna/Vol01_BS/Res36x36",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMAFIT_Vienna/Vol01_BS/Res50x50",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMAFIT_Vienna/Vol01_BS/Res64x64x41",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMA_Vienna/Vol01_WB/Res36x36",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMA_Vienna/Vol01_WB/Res50x50",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/PRISMA_Vienna/Vol01_WB/Res64x64x41",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/VIDA_Vienna/Vol01_PW/Res36x36",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/VIDA_Vienna/Vol01_PW/Res50x50",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/B0corrected_wo_LipidMask/VIDA_Vienna/Vol01_PW/Res64x64x41"    
]

# Load water, lipid and metabo data

In [ ]:
# Load every resolution independently; spatial dimensions are not stacked.
scaling_subjects = load_scaling_calibration_subjects(
    SUBJECT_DIRS,
    original_relative_path="OriginalData/data.npy",
    after_walinet_relative_path="OriginalData/data_after_walinet.npy",
    resource_relative_path=(
        "TrainData/SimulationResources_water_lipid_v1.h5"
    ),
    verify_external_mask=True,
)


# Compute Lipd / Metabo, Water / Metabo tensor ratios

In [ ]:
n_points_by_subject = {
    subject.water_fids.shape[-1]
    for subject in scaling_subjects
}

if len(n_points_by_subject) != 1:
    raise ValueError(
        "Subjects have different FID lengths: "
        f"{sorted(n_points_by_subject)}"
    )

n_points = n_points_by_subject.pop()

frequency_hz = np.fft.fftshift(
    np.fft.fftfreq(
        n_points,
        d=1.0 / bandwidth_hz,
    )
)

ppm = (
    water_ppm
    - frequency_hz
    / (nmr_frequency_hz / 1e6)
)

water_ratio_pools_by_subject = []
lipid_ratio_pools_by_subject = []

for subject in scaling_subjects:
    water_ratio, lipid_ratio = calculate_component_ratios(
        Water=subject.water_fids,
        Lipids=subject.lipids,
        Metabos=subject.metabolites,
        brain_mask=subject.brain_mask,
        ppm=ppm,
        spectral_axis=-1,
        lipid_ppm_max=4.0,
    )

    print(
        subject.subject_dir.name,
        "water ratio:",
        water_ratio.shape,
        "lipid ratio:",
        lipid_ratio.shape,
    )

    water_ratio_pools_by_subject.append(
        pool_valid_voxels(
            water_ratio,
            subject.brain_mask,
        )
    )

    lipid_ratio_pools_by_subject.append(
        pool_valid_voxels(
            lipid_ratio,
            subject.brain_mask,
        )
    )

water_ratio_pool = np.concatenate(
    water_ratio_pools_by_subject
)

lipid_ratio_pool = np.concatenate(
    lipid_ratio_pools_by_subject
)

print("Pooled water ratios:", water_ratio_pool.shape)
print("Pooled lipid ratios:", lipid_ratio_pool.shape)

In [ ]:
water_median, water_iqr = calculate_pooled_median_iqr(
    water_ratio_pool
)

lipid_median, lipid_iqr = calculate_pooled_median_iqr(
    lipid_ratio_pool
)

print(
    f"Water / Metabolites: median={water_median:.3f}, "
    f"IQR={water_iqr:.3f}"
)

print(
    f"Lipids / Metabolites: median={lipid_median:.3f}, "
    f"IQR={lipid_iqr:.3f}"
)


In [ ]:
water_model = compare_positive_models(
    water_ratio_pool,
    title="Water / Metabolites",
    xlabel="Water / Metabolite ratio",
    bins=80,
    truncated_normal_sigma_factor=2.0,
    lognormal_sigma_factor=2.0,
    plot_percentile=99.5,
)

lipid_model = compare_positive_models(
    lipid_ratio_pool,
    title="Lipid / Metabolites",
    xlabel="Lipid / Metabolite ratio",
    bins=80,
    truncated_normal_sigma_factor=2.0,
    lognormal_sigma_factor=2.0,
    plot_percentile=99.5,
)